# ArborGrid · 03 Ouro dimensional (Requisito 07)

**Grão do fato principal (`ouro_fato_risco_arboreo`): uma linha por árvore por dia de referência,
com o risco previsto para as 48 horas seguintes.** A árvore entra no fato a partir do dia em que o
software a detectou.

| Tabela | Tipo | Grão |
|---|---|---|
| `ouro_fato_risco_arboreo` | Fato | árvore × dia (risco das 48h seguintes) |
| `ouro_fato_clima_hora` | Fato | região × hora |
| `ouro_fato_denuncia` | Fato | uma denúncia (dados pessoais protegidos) |
| `ouro_dim_arvore` | Dimensão | uma árvore do GeoSampa |
| `ouro_dim_regiao` | Dimensão | uma das 4 regiões da API ArborGrid |
| `ouro_dim_data` | Dimensão | um dia |

**Score de risco (0–100)** = `risco_arvore × fator_vento`, limitado a 100
- `risco_arvore`: triagem CRITICO 70 · PROXIMIDADE 40 · SEGURO 10; **+15** se há contato direto galho × fio;
  **+15** se nunca houve vistoria oficial ou a última tem mais de 10 anos
- `fator_vento` pela maior rajada prevista nas 48h: < 30 km/h ×0,6 · 30–49 ×0,8 · 50–69 ×1,0 · ≥ 70 ×1,2
- Prioridade (mesmos níveis da API ArborGrid): **URGENTE** ≥ 85 · **ALTA** 65–84 · **MEDIA** 40–64 · **BAIXA** < 40

In [0]:
SET TIME ZONE 'America/Sao_Paulo';

key,value
spark.sql.session.timeZone,America/Sao_Paulo


## Dimensões

In [0]:
CREATE OR REPLACE TABLE workspace.gs.ouro_dim_regiao AS
SELECT * FROM VALUES
  ('pinheiros',        'Pinheiros (Zona Oeste)',                         -23.5630, -46.6857, false),
  ('alto_da_mooca',    'Alto da Mooca (Zona Leste)',                     -23.5616, -46.5928, false),
  ('vila_mariana',     'Vila Mariana (Zona Sul)',                        -23.5890, -46.6350, false),
  ('avenida_paulista', 'Avenida Paulista (controle: rede subterrânea)',   -23.5614, -46.6558, true)
AS t(region_key, regiao, lat_centro, lon_centro, grupo_controle);

CREATE OR REPLACE TABLE workspace.gs.ouro_dim_data AS
WITH datas AS (
  SELECT CAST(data_hora AS DATE) AS data FROM workspace.gs.prata_clima
  UNION SELECT CAST(data_hora_captura AS DATE) FROM workspace.gs.prata_arvores
  UNION SELECT CAST(data_hora_relato AS DATE) FROM workspace.gs.prata_denuncias
)
SELECT DISTINCT
  CAST(date_format(data, 'yyyyMMdd') AS INT) AS sk_data,
  data,
  CASE dayofweek(data) WHEN 1 THEN 'domingo' WHEN 2 THEN 'segunda' WHEN 3 THEN 'terça' WHEN 4 THEN 'quarta'
                       WHEN 5 THEN 'quinta' WHEN 6 THEN 'sexta' ELSE 'sábado' END AS dia_semana,
  month(data) AS mes,
  year(data)  AS ano
FROM datas;

CREATE OR REPLACE TABLE workspace.gs.ouro_dim_arvore AS
WITH por_arvore AS (
  SELECT
    geosampa_id,
    max_by(region_key, data_hora_captura)              AS region_key,
    max_by(distrito, data_hora_captura)                AS distrito,
    max_by(subprefeitura, data_hora_captura)           AS subprefeitura,
    ROUND(AVG(latitude), 6)                            AS latitude,
    ROUND(AVG(longitude), 6)                           AS longitude,
    MIN(CAST(data_hora_captura AS DATE))               AS data_deteccao,
    MAX(ultima_vistoria_sisgau)                        AS ultima_vistoria_sisgau,
    max_by(triagem_arvore, data_hora_captura)          AS triagem_arvore,
    max_by(triagem_arvore_detalhe, data_hora_captura)  AS triagem_arvore_detalhe,
    COUNT(*)                                           AS n_vistas,
    SUM(CASE WHEN triagem_foto = 'CRITICO' THEN 1 ELSE 0 END) AS n_vistas_criticas,
    MAX(contact_direct_px)                             AS contato_direto_px_max,
    MAX(wind_buffer_px)                                AS wind_buffer_px_max,
    ROUND(AVG(confianca), 2)                           AS confianca_media,
    max_by(model_version, data_hora_captura)           AS model_version
  FROM workspace.gs.prata_arvores
  GROUP BY geosampa_id
)
SELECT *,
  CAST(floor(datediff(current_date(), ultima_vistoria_sisgau) / 365.25) AS INT) AS anos_desde_vistoria,
  LEAST(100,
      CASE triagem_arvore WHEN 'CRITICO' THEN 70 WHEN 'PROXIMIDADE' THEN 40 ELSE 10 END
    + CASE WHEN contato_direto_px_max > 0 THEN 15 ELSE 0 END
    + CASE WHEN ultima_vistoria_sisgau IS NULL
             OR datediff(current_date(), ultima_vistoria_sisgau) > 3652 THEN 15 ELSE 0 END
  ) AS risco_arvore
FROM por_arvore;

num_affected_rows,num_inserted_rows


## Fatos

In [0]:
CREATE OR REPLACE TABLE workspace.gs.ouro_fato_clima_hora AS
SELECT
  CAST(date_format(data_hora, 'yyyyMMdd') AS INT) AS sk_data,
  region_key,
  data_hora,
  hour(data_hora)  AS hora,
  rajada_kmh,
  vento_kmh,
  chuva_mm,
  rajada_kmh >= 50 AS vento_forte
FROM workspace.gs.prata_clima;

CREATE OR REPLACE TABLE workspace.gs.ouro_fato_risco_arboreo AS
WITH dias AS (  -- dias de referência: até hoje, e com as 48h seguintes de previsão disponíveis
  SELECT DISTINCT CAST(data_hora AS DATE) AS data
  FROM workspace.gs.prata_clima
  WHERE CAST(data_hora AS DATE) <= current_date()
    AND date_add(CAST(data_hora AS DATE), 1) <= (SELECT MAX(CAST(data_hora AS DATE)) FROM workspace.gs.prata_clima)
),
janela AS (     -- clima previsto nas 48h a partir de cada dia, por região
  SELECT d.data, c.region_key,
         MAX(c.rajada_kmh)                                       AS rajada_max_48h,
         max_by(c.data_hora, c.rajada_kmh)                       AS hora_pico_rajada,
         ROUND(SUM(c.chuva_mm), 1)                               AS chuva_48h_mm,
         SUM(CASE WHEN c.rajada_kmh >= 50 THEN 1 ELSE 0 END)     AS horas_vento_forte_48h
  FROM dias d
  JOIN workspace.gs.prata_clima c
    ON c.data_hora >= CAST(d.data AS TIMESTAMP)
   AND c.data_hora <  CAST(date_add(d.data, 2) AS TIMESTAMP)
  GROUP BY d.data, c.region_key
),
base AS (
  SELECT
    CAST(date_format(j.data, 'yyyyMMdd') AS INT) AS sk_data,
    a.geosampa_id, a.region_key,
    a.n_vistas, a.n_vistas_criticas, a.risco_arvore,
    j.rajada_max_48h, j.hora_pico_rajada, j.chuva_48h_mm, j.horas_vento_forte_48h,
    CASE WHEN j.rajada_max_48h >= 70 THEN 1.2 WHEN j.rajada_max_48h >= 50 THEN 1.0
         WHEN j.rajada_max_48h >= 30 THEN 0.8 ELSE 0.6 END AS fator_vento
  FROM workspace.gs.ouro_dim_arvore a
  JOIN janela j ON j.region_key = a.region_key AND a.data_deteccao <= j.data
),
pontuado AS (
  SELECT *, ROUND(LEAST(100.0, risco_arvore * fator_vento), 1) AS composite_risk_score
  FROM base
)
SELECT *,
  CASE WHEN composite_risk_score >= 85 THEN 'URGENTE' WHEN composite_risk_score >= 65 THEN 'ALTA'
       WHEN composite_risk_score >= 40 THEN 'MEDIA' ELSE 'BAIXA' END AS priority_tier,
  CASE WHEN composite_risk_score >= 85 THEN 'Poda emergencial antes da rajada prevista; acionar Defesa Civil e Enel'
       WHEN composite_risk_score >= 65 THEN 'Agendar poda prioritária em até 7 dias'
       WHEN composite_risk_score >= 40 THEN 'Vistoria técnica no próximo ciclo'
       ELSE 'Manter monitoramento cíclico de rotina' END AS recommended_action
FROM pontuado;

num_affected_rows,num_inserted_rows


## Denúncias na Ouro com dados pessoais protegidos (LGPD)

| Dado pessoal | Bronze | Prata | Ouro |
|---|---|---|---|
| Nome do morador | sim | **não** (minimização) | não |
| E-mail | sim | sim | **hash** SHA-256 com sal (conta reincidência sem identificar) |
| Telefone | sim | sim | **máscara** `(11) *****-1234` |
| GPS do celular | sim | sim | **generalizado** para 3 casas decimais (~100 m) |

O **sal** fica fora do código: secret scope `arborgrid` / chave `sal_lgpd` ou, se o workspace não tiver secret scope, o arquivo `_controle/sal.txt` no Volume.

In [0]:
%python
from pyspark.sql import functions as F

# O sal fica FORA do código: secret scope (se existir) ou arquivo no Volume (_controle/sal.txt)
try:
    SAL = dbutils.secrets.get(scope="arborgrid", key="sal_lgpd")
except Exception:
    with open("/Volumes/workspace/gs/chegada/_controle/sal.txt", encoding="utf-8") as f:
        SAL = f.read().strip()
assert SAL, "O arquivo _controle/sal.txt está vazio."

den = spark.table("workspace.gs.prata_denuncias")
fato_denuncia = den.select(
    F.date_format(F.to_date("data_hora_relato"), "yyyyMMdd").cast("int").alias("sk_data"),
    "id_denuncia", "region_key", "tipo",
    "data_hora_relato", "data_hora_envio", "atraso_envio_min",
    F.round("latitude", 3).alias("lat_aprox"),
    F.round("longitude", 3).alias("lon_aprox"),
    F.sha2(F.concat(F.lower(F.trim("reportante_email")), F.lit(SAL)), 256).alias("reportante_hash"),
    F.regexp_replace("reportante_telefone", r"^\((\d{2})\) ?\d{4,5}-(\d{4})$", r"($1) *****-$2")
        .alias("telefone_mascarado"),
    "foto_url",
)
(fato_denuncia.write.mode("overwrite").option("overwriteSchema", True)
    .saveAsTable("workspace.gs.ouro_fato_denuncia"))
display(spark.table("workspace.gs.ouro_fato_denuncia").limit(5))

sk_data,id_denuncia,region_key,tipo,data_hora_relato,data_hora_envio,atraso_envio_min,lat_aprox,lon_aprox,reportante_hash,telefone_mascarado,foto_url
20261110,DN-20261110-003,alto_da_mooca,fio_partido,2026-11-10T19:39:00.000-03:00,2026-11-11T01:39:00.000-03:00,360,-23.569,-46.593,e786ace5196074957366bb05249e83247f95bf2f34cb08705f732870d5d1a496,(11) *****-4188,https://app.arborgrid.example/fotos/2026-11-10/003.jpg
20261110,DN-20261110-004,alto_da_mooca,fio_partido,2026-11-10T18:47:00.000-03:00,2026-11-10T23:47:00.000-03:00,300,-23.557,-46.596,a84560bc116601eaf64e5128840873fa604fb5ec09e121b47cf9c6bc3dfdee0d,(11) *****-9682,https://app.arborgrid.example/fotos/2026-11-10/004.jpg
20261110,DN-20261110-005,pinheiros,fio_partido,2026-11-10T06:04:00.000-03:00,2026-11-10T06:04:27.000-03:00,0,-23.555,-46.684,91c689db95f8cfc35b696afe917d39cab61ac2c0fd5e337c47fa1977acce2c4e,(11) *****-9020,https://app.arborgrid.example/fotos/2026-11-10/005.jpg
20261110,DN-20261110-006,pinheiros,fio_partido,2026-11-10T07:16:00.000-03:00,2026-11-10T07:16:13.000-03:00,0,-23.568,-46.689,91c689db95f8cfc35b696afe917d39cab61ac2c0fd5e337c47fa1977acce2c4e,(11) *****-9020,https://app.arborgrid.example/fotos/2026-11-10/006.jpg
20261110,DN-20261110-007,pinheiros,arvore_seca,2026-11-10T16:52:00.000-03:00,2026-11-10T16:52:08.000-03:00,0,-23.557,-46.688,8f9051a2733292984f77c5ba4ffaad6d97727b3d674b766b6387388b03272207,(11) *****-8484,https://app.arborgrid.example/fotos/2026-11-10/007.jpg


## Integridade (print para o relatório)

In [0]:
WITH ult AS (SELECT MAX(sk_data) AS sk FROM workspace.gs.ouro_fato_risco_arboreo)
SELECT 'dim_arvore: chave única' AS verificacao,
       COUNT(*) = COUNT(DISTINCT geosampa_id) AS ok, COUNT(*) AS valor
FROM workspace.gs.ouro_dim_arvore
UNION ALL
SELECT 'dim_regiao: chave única', COUNT(*) = COUNT(DISTINCT region_key), COUNT(*)
FROM workspace.gs.ouro_dim_regiao
UNION ALL
SELECT 'dim_data: chave única', COUNT(*) = COUNT(DISTINCT sk_data), COUNT(*)
FROM workspace.gs.ouro_dim_data
UNION ALL
SELECT 'fato_risco (último dia): soma de vistas = linhas da prata_arvores',
       (SELECT SUM(n_vistas) FROM workspace.gs.ouro_fato_risco_arboreo, ult WHERE sk_data = ult.sk)
         = (SELECT COUNT(*) FROM workspace.gs.prata_arvores),
       (SELECT SUM(n_vistas) FROM workspace.gs.ouro_fato_risco_arboreo, ult WHERE sk_data = ult.sk)
UNION ALL
SELECT 'fato_risco (último dia): uma linha por árvore',
       (SELECT COUNT(*) FROM workspace.gs.ouro_fato_risco_arboreo, ult WHERE sk_data = ult.sk)
         = (SELECT COUNT(*) FROM workspace.gs.ouro_dim_arvore),
       (SELECT COUNT(*) FROM workspace.gs.ouro_fato_risco_arboreo, ult WHERE sk_data = ult.sk)
UNION ALL
SELECT 'fato_clima_hora: soma da chuva = prata_clima',
       (SELECT ROUND(SUM(chuva_mm), 2) FROM workspace.gs.ouro_fato_clima_hora)
         = (SELECT ROUND(SUM(chuva_mm), 2) FROM workspace.gs.prata_clima),
       CAST((SELECT ROUND(SUM(chuva_mm), 2) FROM workspace.gs.ouro_fato_clima_hora) AS BIGINT)
UNION ALL
SELECT 'fato_clima_hora: linhas = prata_clima',
       (SELECT COUNT(*) FROM workspace.gs.ouro_fato_clima_hora) = (SELECT COUNT(*) FROM workspace.gs.prata_clima),
       (SELECT COUNT(*) FROM workspace.gs.ouro_fato_clima_hora)
UNION ALL
SELECT 'fato_denuncia: linhas = prata_denuncias',
       (SELECT COUNT(*) FROM workspace.gs.ouro_fato_denuncia) = (SELECT COUNT(*) FROM workspace.gs.prata_denuncias),
       (SELECT COUNT(*) FROM workspace.gs.ouro_fato_denuncia)
UNION ALL
SELECT 'fato_risco: toda árvore existe na dim_arvore', COUNT(*) = 0, COUNT(*)
FROM workspace.gs.ouro_fato_risco_arboreo f
LEFT ANTI JOIN workspace.gs.ouro_dim_arvore a ON f.geosampa_id = a.geosampa_id
UNION ALL
SELECT 'fato_risco: toda data existe na dim_data', COUNT(*) = 0, COUNT(*)
FROM workspace.gs.ouro_fato_risco_arboreo f
LEFT ANTI JOIN workspace.gs.ouro_dim_data d ON f.sk_data = d.sk_data;

verificacao,ok,valor
dim_arvore: chave única,true,262
dim_regiao: chave única,true,4
dim_data: chave única,true,10
fato_risco (último dia): soma de vistas = linhas da prata_arvores,true,505
fato_risco (último dia): uma linha por árvore,true,262
fato_clima_hora: soma da chuva = prata_clima,true,145
fato_clima_hora: linhas = prata_clima,true,960
fato_denuncia: linhas = prata_denuncias,true,62
fato_risco: toda árvore existe na dim_arvore,true,0
fato_risco: toda data existe na dim_data,true,0
